# Tutorial: Palette catalog summaries

## Audience
- Python learners who want to summarize a CSV catalog using only the standard library.

## Prerequisites
- Basic Python: functions, dictionaries, lists and sorting.
- A UTF-8 CSV with `Primary (Hex)` and `Product Type` headers.

## Learning goals
- Read a CSV by header names with `csv.DictReader`.
- Normalize text values before grouping.
- Build deterministic, sorted summaries.


## Outline

1. Setup
2. Define `summarize_palettes`
3. Summarize the catalog
4. Exercises


In [ ]:
# Setup: standard library only
from __future__ import annotations

import csv
from collections import defaultdict

CATALOG_PATH = '/inputs/colors.csv'


## Step 1 - Group rows by normalized primary color

Each row's `Primary (Hex)` is stripped and uppercased; `Product Type` is stripped.
Every row counts (duplicates included); product types are de-duplicated and sorted.
Groups are ordered by count descending, then primary ascending.


In [ ]:
def summarize_palettes(csv_path):
    counts = defaultdict(int)
    types = defaultdict(set)
    with open(csv_path, encoding='utf-8', newline='') as handle:
        for row in csv.DictReader(handle):
            primary = (row['Primary (Hex)'] or '').strip().upper()
            product_type = (row['Product Type'] or '').strip()
            counts[primary] += 1
            types[primary].add(product_type)
    groups = [
        {'primary': primary, 'count': counts[primary], 'product_types': sorted(types[primary])}
        for primary in counts
    ]
    groups.sort(key=lambda group: (-group['count'], group['primary']))
    return groups


## Step 2 - Summarize the catalog

Apply the function to the real catalog and look at the largest groups.


In [ ]:
palette_summary = summarize_palettes(CATALOG_PATH)
[(group['primary'], group['count']) for group in palette_summary[:5]]


## Exercises

- Group by `CTA (Hex)` instead of the primary color.
- Predict what an empty (header-only) catalog returns, then check.
- Common mistake: forgetting to normalize case, which splits `#2563eb` and `#2563EB` into two groups.


In [ ]:
# Exercise scaffold: quick sanity checks on the summary
with open(CATALOG_PATH, encoding='utf-8', newline='') as handle:
    total_rows = sum(1 for _ in csv.DictReader(handle))
assert sum(group['count'] for group in palette_summary) == total_rows
len(palette_summary)
